<a href="https://colab.research.google.com/github/aelamin25/Monkey-AGI/blob/main/qta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Can a fine tuned BERT model, trained on ACLED text data of air/drone strikes across, predict casualty severty in Ukraine? And how does it's performance compare to LLM classifications (Zero-shot and upwards)?

Setting up the environment by installin relevant Python packages

In [ ]:
!pip install transformers
!pip install datasets
!pip install tqdm
!pip install scikit-learn

In [ ]:
# This just ensures that the text is wrapped in the output display of Google colab...

from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

Now i will import neccessary libraries

In [ ]:
# Hugging face transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from transformers.pipelines.pt_utils import KeyDataset
import datasets

# Progress bar
from tqdm.auto import tqdm

#Pytorch
import torch

# Data handling
import pandas as pd
import numpy as np
import re # this will help us remove the "casualty" language from our ACLED data

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


Import data

In [ ]:
# Note you have to add the files to the colab session (go to file icon on the left --> tap arrow pointing upwards under Files titles --> import the files from desktop)

data_df = pd.read_csv("/content/Only Ukr 22-25.csv")

See my data

In [ ]:
data_df .describe()

,year,time_precision,iso,latitude,longitude,geo_precision,fatalities,timestamp,population_1km,population_2km,population_5km,population_best
count,6250.000000,6250.000000,6250.0,6250.000000,6250.000000,6250.000000,6250.000000,6.250000e+03,6248.000000,6248.000000,6248.000000,6248.000000
mean,2023.351040,1.036960,804.0,48.453390,36.194804,1.754720,1.687200,1.756027e+09,1357.961268,4222.354193,13719.874360,13719.874360
std,0.890178,0.192041,0.0,1.411668,2.068010,0.476859,8.698663,6.336255e+06,2118.765193,6946.452694,31224.580696,31224.580696
min,2022.000000,1.000000,804.0,43.389000,23.385200,1.000000,0.000000,1.647377e+09,0.000000,0.000000,0.000000,0.000000
25%,2023.000000,1.000000,804.0,47.567300,34.973200,1.000000,0.000000,1.753956e+09,122.000000,237.000000,323.000000,323.000000
50%,2023.000000,1.000000,804.0,48.281000,37.016550,2.000000,0.000000,1.753956e+09,364.500000,827.000000,1210.000000,1210.000000
75%,2024.000000,1.000000,804.0,49.270175,37.848300,2.000000,2.000000,1.755647e+09,1632.000000,5917.500000,10266.000000,10266.000000
max,2025.000000,3.000000,804.0,52.282000,39.644900,3.000000,300.000000,1.774378e+09,19452.000000,41805.000000,212551.000000,212551.000000


Create the binary labels (I.e were there casualties or not)

In [ ]:
data_df ['Label'] = (data_df ['fatalities'] > 0).astype(int) # 1 if Casuailites, 0 otherwise


Let us see the distribution (how much of my data includes Casualities (CT)

In [ ]:
print(f"Data label distribution: {data_df['Label'].value_counts()}") # The 'f' allows us to include variables (i.e text and variables)

# Split as a propotion
print(f"Data label distribution: {data_df['Label'].value_counts(normalize=True)}")

Data label distribution: Label
0    3493
1    2757
Name: count, dtype: int64
Data label distribution: Label
0    0.55888
1    0.44112
Name: proportion, dtype: float64


First I need to remove any rows that mention casualties unknown.

In [ ]:
Before_Count = len(data_df )                                        # Track how many rows we drop for the methodology section
data_df  = data_df [~data_df ['notes'].str.contains('casualties unknown', case=False, na=False)].reset_index(drop=True)  # Drop rows containing the exact phrase, case-insensitive
After_Count = len(data_df )

print(f"Dropped {Before_Count - After_Count} events with 'casualties unknown' phrasing")  # Report drop count for transparency
print(f"Data set now: {After_Count} events")

Dropped 2855 events with 'casualties unknown' phrasing
Data set now: 3395 events


Check proportions again

In [ ]:
# Split as a propotion
print(f"Data label distribution: {data_df['Label'].value_counts(normalize=True)}")

Data label distribution: Label
1    0.808542
0    0.191458
Name: proportion, dtype: float64


80% of my data now has Casualty numbers.

Now we need to work on hiding the information that captures Casualty.